# Lab 7 Assignment – Sentiment Analysis: Data-Centric vs Model-Centric AI

**Name:** Nafeesa Mahek  
**ID:** S23108186  
**Course:** Machine Learning  
**Date:** May 2026

---

## Setup – Install & Import Libraries

In [ ]:
!pip install scikit-learn pandas --quiet

In [ ]:
import pandas as pd
import numpy as np
from sklearn import metrics, clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

---

## Load the Data

In [ ]:
train = pd.read_csv('reviews_train.csv')
test  = pd.read_csv('reviews_test.csv')

print(f"Training samples : {len(train)}")
print(f"Test samples     : {len(test)}")
test.sample(5)

---

## Part 1 – Function Design (Core Task)

### Task 1: Build a Model Comparison Function

The function below accepts a list of `(name, model)` tuples, training data, and test data. For each model it:
1. Trains the model on the training set
2. Predicts on the test set
3. Computes Accuracy, Precision, Recall, and F1-score

It returns a tidy **DataFrame** so results are easy to compare at a glance.

In [ ]:
def compare_models(models, X_train, y_train, X_test, y_test):
    """
    Train and evaluate multiple machine learning models.

    Parameters
    ----------
    models : list of (str, estimator) tuples  OR  dict {name: estimator}
        The models to compare. Each estimator must implement .fit() and .predict().
    X_train : array-like
        Training features.
    y_train : array-like
        Training labels.
    X_test : array-like
        Test features.
    y_test : array-like
        True test labels.

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns: Model, Accuracy, Precision, Recall, F1-Score.
    """
    # Accept either a dict or a list of (name, model) pairs
    if isinstance(models, dict):
        models = list(models.items())

    results = []

    for name, model in models:
        # Step 1 – Train
        model.fit(X_train, y_train)

        # Step 2 – Predict
        y_pred = model.predict(X_test)

        # Step 3 – Evaluate
        acc       = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, pos_label='good', zero_division=0)
        recall    = recall_score(y_test, y_pred,    pos_label='good', zero_division=0)
        f1        = f1_score(y_test, y_pred,        pos_label='good', zero_division=0)

        results.append({
            'Model'    : name,
            'Accuracy' : round(acc,       4),
            'Precision': round(precision, 4),
            'Recall'   : round(recall,    4),
            'F1-Score' : round(f1,        4),
        })

    return pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)

#### Demonstrating the function on the raw (uncleaned) training data

We build text pipelines for three classifiers and pass them to `compare_models`.

In [ ]:
# Define candidate models as (name, sklearn Pipeline) pairs
candidate_models = [
    (
        'SGD (SVM)',
        Pipeline([
            ('vect',  CountVectorizer()),
            ('tfidf', TfidfTransformer()),
            ('clf',   SGDClassifier(random_state=42)),
        ])
    ),
    (
        'Naive Bayes',
        Pipeline([
            ('vect',  CountVectorizer()),
            ('tfidf', TfidfTransformer()),
            ('clf',   MultinomialNB()),
        ])
    ),
    (
        'Logistic Regression',
        Pipeline([
            ('vect',  CountVectorizer(ngram_range=(1, 2))),
            ('tfidf', TfidfTransformer()),
            ('clf',   LogisticRegression(C=0.01, random_state=42, max_iter=1000)),
        ])
    ),
]

# Run comparison on the raw (noisy) data
results_raw = compare_models(
    candidate_models,
    X_train=train['review'], y_train=train['label'],
    X_test=test['review'],   y_test=test['label']
)

print("=== Model Comparison on RAW (uncleaned) training data ===")
results_raw

---

## Task 2 – Explore the Data and Identify Bad Data Points

Before we can improve results through data cleaning, we need to understand what is wrong with the dataset.  
Let's inspect a few examples.

In [ ]:
# Look at the first few rows
train.head()

In [ ]:
# Zoom in on the very first data point
print(train.iloc[0].to_dict())

The first entry is labelled **"good"** but the review text is clearly **negative**. It also contains raw HTML tags like `<br>` at the end — a sign the data was scraped and not cleaned properly.

In [ ]:
# Find all rows that contain HTML tags
html_mask  = train['review'].str.contains('<.*?>', regex=True, na=False)
weird_data = train[html_mask]

print(f"Total training rows  : {len(train)}")
print(f"Rows with HTML tags  : {len(weird_data)}")
print(f"Percentage affected  : {100*len(weird_data)/len(train):.1f}%\n")
weird_data.head(3)

### Observations (Exercise 2 – Answer)

After examining the data, here is the pattern I noticed:

- **Bad data points all contain stray HTML tags** (e.g., `<br>`, `</tr>`, `<p>`, etc.). These appear at the end of the review text and are remnants of poor HTML scraping.
- **The labels for these rows are completely flipped.** A review that reads as very negative is labelled `good`, and a clearly positive review is labelled `bad`.
- The HTML presence is therefore a **reliable signal** for a mislabelled (noisy) data point.
- Clean rows (no HTML) appear to have correct labels and normal English text.

---

## Task 3 – Clean the Data and Retrain

A simple but effective heuristic: **any review that contains both `<` and `>` is considered bad data and is removed.**

In [ ]:
def is_bad_data(review: str) -> bool:
    """
    Return True if the review contains HTML-like tags (bad/mislabelled data).

    The heuristic: if both '<' and '>' appear in the text, it likely
    contains residual HTML from a scraping error, which also means the
    label is inverted and the row should be discarded.
    """
    review_str = str(review)
    return '<' in review_str and '>' in review_str

In [ ]:
# Apply the heuristic to filter out bad rows
train_clean = train[~train['review'].map(is_bad_data)].copy()

print(f"Original training size : {len(train)}")
print(f"Cleaned training size  : {len(train_clean)}")
print(f"Rows removed           : {len(train) - len(train_clean)}")

### Retrain all models on the cleaned data using `compare_models`

We rebuild fresh pipelines (so no fitted state leaks from the earlier run) and call our comparison function on the cleaned training set.

In [ ]:
clean_models = [
    (
        'SGD (SVM)',
        Pipeline([
            ('vect',  CountVectorizer()),
            ('tfidf', TfidfTransformer()),
            ('clf',   SGDClassifier(random_state=42)),
        ])
    ),
    (
        'Naive Bayes',
        Pipeline([
            ('vect',  CountVectorizer()),
            ('tfidf', TfidfTransformer()),
            ('clf',   MultinomialNB()),
        ])
    ),
    (
        'Logistic Regression',
        Pipeline([
            ('vect',  CountVectorizer(ngram_range=(1, 2))),
            ('tfidf', TfidfTransformer()),
            ('clf',   LogisticRegression(C=0.01, random_state=42, max_iter=1000)),
        ])
    ),
]

results_clean = compare_models(
    clean_models,
    X_train=train_clean['review'], y_train=train_clean['label'],
    X_test=test['review'],         y_test=test['label']
)

print("=== Model Comparison on CLEANED training data ===")
results_clean

### Before vs After – Side-by-Side Comparison

In [ ]:
# Merge the two result tables for a clean comparison
comparison = results_raw.merge(
    results_clean,
    on='Model',
    suffixes=(' (raw)', ' (clean)')
).sort_values('F1-Score (clean)', ascending=False).reset_index(drop=True)

print("=== Raw vs Cleaned – All Metrics ===")
comparison

### Conclusion

By simply **removing the mislabelled rows that contained HTML tags**, every model saw a significant improvement in all metrics — with no changes to model architecture or hyperparameters. This illustrates the core idea of **Data-Centric AI**: investing effort in data quality rather than model complexity often yields the biggest gains.

| Takeaway | Detail |
|---|---|
| Root cause | HTML scraping artifacts caused label flipping |
| Fix | Filter rows where `review` contains `<` and `>` |
| Result | All three models improved substantially after cleaning |